# Misinformation Models  — Opinion Classification
Driver notebook: runs each model, collects predictions, compares metrics.  
Add new models in the **Run models** cell. Train/dev only — test set not touched.
Had help from Claude on implementation.

### Import configuration and packages ###

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))

import pandas as pd
import numpy as np
import torch
import cnn_baseline as cnn

#Running this will import FastText vector file, which is stored on HuggingFace and is >4gb. 
from config import DATA_DIR, FASTTEXT_PATH, TARGETS

from preprocess import preprocess
from metrics import compute_metrics, print_confusion_matrix, print_sklearn_report, error_analysis, print_report

DEVICE = torch.device(
    "mps"  if torch.backends.mps.is_available()  else
    "cuda" if torch.cuda.is_available()           else
    "cpu"
)
print(f"Device: {DEVICE}")
print(f"Targets: {TARGETS}")

/home/nicole-shantz/miniforge3/envs/colx_misinformation/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cuda
Targets: ['opinion_label', 'misinformation_label']


In [2]:
# Load data (shared across all models)
train_rows = cnn.load_csv(DATA_DIR / "mis_df_train.csv")
dev_rows   = cnn.load_csv(DATA_DIR / "mis_df_dev.csv")
print(f"Train: {len(train_rows)} | Dev: {len(dev_rows)}")

Train: 600 | Dev: 200


### Run models ###


In [3]:
# Each model should produce (preds, labels, probs) and store them in `results`.
# Add new models below following the same pattern.
results = {}

In [4]:
# Run CNN model here
# TextCNN — trains on both targets simultaneously, then evaluates each separately
vocab        = cnn.build_vocab(train_rows, preprocess)
embed_matrix = cnn.load_fasttext_vectors(FASTTEXT_PATH, vocab)
train_loader = cnn.make_loader(train_rows, vocab, shuffle=True,  tokenize_fn=preprocess)
dev_loader   = cnn.make_loader(dev_rows,   vocab, shuffle=False, tokenize_fn=preprocess)
cnn_model    = cnn.TextCNN(len(vocab), embed_matrix).to(DEVICE)
cnn_model    = cnn.train_model(cnn_model, train_loader, dev_loader, train_rows, DEVICE)

for target in TARGETS:
    results[f"TextCNN — {target}"] = cnn.predict(cnn_model, dev_loader, DEVICE, target=target)

Loading FastText vectors from /home/nicole-shantz/.cache/huggingface/hub/datasets--COLX523--fasttext-cc-en-300/snapshots/63ba06b23eb770a6fe0f6c17b0de981e43aec6aa/cc.en.300.vec …
  3875/4091 vocab tokens found in FastText vectors (94.7%)
Epoch   1 | loss=1.7817 | avg_dev_f1=0.6906 (opinion_label: 0.6779 | misinformation_label: 0.7032)
Epoch   2 | loss=1.5527 | avg_dev_f1=0.7172 (opinion_label: 0.6650 | misinformation_label: 0.7693)
Epoch   3 | loss=1.3878 | avg_dev_f1=0.7362 (opinion_label: 0.6985 | misinformation_label: 0.7738)
Epoch   4 | loss=1.1971 | avg_dev_f1=0.7425 (opinion_label: 0.6827 | misinformation_label: 0.8022)
Epoch   5 | loss=1.0453 | avg_dev_f1=0.7426 (opinion_label: 0.6748 | misinformation_label: 0.8104)
Epoch   6 | loss=0.9247 | avg_dev_f1=0.7668 (opinion_label: 0.6970 | misinformation_label: 0.8366)
Epoch   7 | loss=0.8254 | avg_dev_f1=0.7495 (opinion_label: 0.6746 | misinformation_label: 0.8245)
Epoch   8 | loss=0.7510 | avg_dev_f1=0.7588 (opinion_label: 0.6624 | m

In [5]:
# Run Logistic Regression model — trained separately per target
import logreg_baseline as lr

for target in TARGETS:
    results[f"LogReg — {target}"] = lr.run(train_rows, dev_rows, task=target)


── Logistic Regression  [opinion_label] ──
  Building features (TF-IDF + linguistic)
  Feature matrix: train=(600, 2504), dev=(200, 2504)
  Running GridSearchCV over C
  Best C: 1.0  |  CV macro-F1: 0.7013

── Logistic Regression  [misinformation_label] ──
  Building features (TF-IDF + linguistic)
  Feature matrix: train=(600, 2504), dev=(200, 2504)
  Running GridSearchCV over C
  Best C: 1.0  |  CV macro-F1: 0.8514


In [ ]:
# Run Logistic Regression transfer model — trained separately per target
import logreg_transfer as lr_transfer

for target in TARGETS:
    results[f"LogReg (+ embeddings) — {target}"] = lr_transfer.run(train_rows, dev_rows, task=target)


── Logistic Regression  [opinion_label] ──
  Building features (TF-IDF, linguistic features, FastText embeddings)


Loading FastText vectors from /home/nicole-shantz/.cache/huggingface/hub/datasets--COLX523--fasttext-cc-en-300/snapshots/63ba06b23eb770a6fe0f6c17b0de981e43aec6aa/cc.en.300.vec …


In [ ]:
#Create table to compare models across metrics
rows = []
for name, (preds, labels, probs) in results.items():
    m = compute_metrics(preds, labels, probs)
    rows.append({"Model": name, "Accuracy": m["accuracy"], "Macro F1": m["macro_f1"],
                 "F1 (not-op)": m["f1_class0"], "F1 (opinion)": m["f1_class1"],
                 "F1.5 (recall-weighted)": m["fbeta_class1"],
                 "AUC-ROC": m.get("auc_roc", float("nan"))})

pd.set_option("display.float_format", "{:.4f}".format)
pd.DataFrame(rows).set_index("Model")

,Accuracy,Macro F1,F1 (not-op),F1 (opinion),F1.5 (recall-weighted),AUC-ROC
Model,,,,,,
TextCNN — opinion_label,0.7100,0.7090,0.7264,0.6915,0.7241,0.7707
TextCNN — misinformation_label,0.9000,0.8667,0.9333,0.8000,0.7879,0.9528
LogReg — opinion_label,0.7000,0.6931,0.7391,0.6471,0.6530,0.7514
LogReg — misinformation_label,0.9100,0.8784,0.9404,0.8163,0.7975,0.9630
LogReg (+ embeddings) — opinion_label,0.7100,0.7076,0.7339,0.6813,0.7052,0.7704
LogReg (+ embeddings) — misinformation_label,0.9150,0.8889,0.9428,0.8350,0.8318,0.9500


In [ ]:
#Model details
for name, (preds, labels, probs) in results.items():
    print(f"\n{'='*50}\n{name}\n{'='*50}")
    print_confusion_matrix(preds, labels)
    print()
    print_sklearn_report(preds, labels)
    print()
    error_analysis(dev_rows, preds, labels)


TextCNN — opinion_label
Confusion matrix (rows=true, cols=predicted):
                 pred=0  pred=1
  true=0 (not-op):    77      40
  true=1 (opinion):   18      65

              precision    recall  f1-score   support

 not-opinion       0.81      0.66      0.73       117
     opinion       0.62      0.78      0.69        83

    accuracy                           0.71       200
   macro avg       0.71      0.72      0.71       200
weighted avg       0.73      0.71      0.71       200


False Positives (predicted opinion, actually not) — 5 shown:
  [17] 'Birds in a blizzard. We put out extra sunflower seeds since their usual food sources just got‚Ä¶ https://www.instagram.c'
  [25] "I'm getting used to seeing a layer of ash on my car each morning - and I'm in AB. We've  only seen a red sun this week. "
  [55] "@jihettly @esd2000 good morning! Yes I'm in the middle of the blizzard. I have plenty of food so I'm happy! Lol. Stay wa"
  [59] 'Re: Hurricane Matthew: All of the @ASUTenni

In [ ]:
# Simple ensembling method
import importlib
import simple_ensemble
importlib.reload(simple_ensemble)
from simple_ensemble import soft_vote

ensemble_rows = []
for task in TARGETS:
    ensemble_results = soft_vote([
        results[f"TextCNN — {task}"],
        results[f"LogReg — {task}"],
        results[f"LogReg (+ embeddings) — {task}"],
    ])
    preds, labels, probs = ensemble_results
    m = compute_metrics(preds, labels, probs)
    ensemble_rows.append({"Model": f"Soft Vote Ensemble — {task}", 
                          "Accuracy": m["accuracy"], 
                          "Macro F1": m["macro_f1"],
                          "F1 (not-op)": m["f1_class0"], 
                          "F1 (opinion)": m["f1_class1"],
                          "F1.5 (recall-weighted)": m["fbeta_class1"],
                          "AUC-ROC": m["auc_roc"]})

pd.set_option("display.float_format", "{:.4f}".format)
pd.DataFrame(ensemble_rows).set_index("Model")

,Accuracy,Macro F1,F1 (not-op),F1 (opinion),F1.5 (recall-weighted),AUC-ROC
Model,,,,,,
Soft Vote Ensemble — opinion_label,0.7450,0.7431,0.7650,0.7213,0.7480,0.7897
Soft Vote Ensemble — misinformation_label,0.9300,0.9090,0.9527,0.8654,0.8654,0.9710


In [ ]:
# Motivational ensembling method
import importlib
import motivated_ensemble
importlib.reload(motivated_ensemble)
from motivated_ensemble import motivated_soft_vote


f1_scores = {
    "opinion_label":        [0.7090, 0.6931, 0.7076],  
    "misinformation_label": [0.8667, 0.8784, 0.8889], 
}

motivated_rows = []
for task in TARGETS:
    preds, labels, probs = motivated_soft_vote(
        model_outputs=[
            results[f"TextCNN — {task}"],
            results[f"LogReg — {task}"],
            results[f"LogReg (+ embeddings) — {task}"],
        ],
        f1_weights=f1_scores[task],
    )
    m = compute_metrics(preds, labels, probs)
    motivated_rows.append({"Model": f"Motivated Ensemble — {task}",
                           "Accuracy": m["accuracy"], "Macro F1": m["macro_f1"],
                           "F1 (not-op)": m["f1_class0"], "F1 (opinion)": m["f1_class1"],
                           "F1.5 (recall-weighted)": m["fbeta_class1"],
                           "AUC-ROC": m["auc_roc"]})

pd.set_option("display.float_format", "{:.4f}".format)
pd.DataFrame(motivated_rows).set_index("Model")